In [0]:
from pyspark.sql.functions import *

In [0]:
bootstrap_servers = dbutils.secrets.get(scope = "finguard-scope", key = "bootstrap_servers")
api_key = dbutils.secrets.get(scope = "finguard-scope", key = "api_key")
api_secret = dbutils.secrets.get(scope = "finguard-scope", key = "api_secret")
topic = dbutils.secrets.get(scope = "finguard-scope", key = "topic")
jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
streaming_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    # .option("endingOffsets", "latest")
    .load()
)

In [0]:
parsed_streaming_df = streaming_df.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)

In [0]:
streaming_query = (
    parsed_streaming_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/finguard/source/transactions/checkpoint/")
    .trigger(availableNow=True)
    .toTable("finguard.bronze.transactions")
)
print("Query started with query id: ", streaming_query.id)

Query started with query id:  91de292a-bb8c-485f-8cdb-23b47aa44766
